<a href="https://colab.research.google.com/github/NxrFesdac/bourbaki-nlp-avanzado/blob/main/modulo5/RAG_Agents_Challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install -U langchain-openai langchain-community langgraph faiss-cpu openai duckdb

In [ ]:
import duckdb

# 1. Create the in-memory DuckDB connection
con = duckdb.connect(database=':memory:')

# 2. Create tables and insert rows
setup_sql = """
-- USERS
CREATE TABLE users (id INT PRIMARY KEY, first_name VARCHAR(50), last_name VARCHAR(50), email VARCHAR(100) UNIQUE, phone VARCHAR(20), created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP);
INSERT INTO users VALUES
(1, 'Ana', 'Gomez', 'cliente@example.com', '555-0101', '2026-02-15 10:00:00'),
(2, 'Carlos', 'Ruiz', 'carlos.r@mail.com', NULL, '2026-03-01 14:30:00'),
(3, 'Beatriz', 'Perez', 'bea.p@mail.com', '555-0103', '2026-03-10 09:15:00'),
(4, 'David', 'Lopes', 'david.l@mail.com', NULL, '2026-01-20 11:20:00'),
(5, 'Elena', 'Torres', 'elena.t@mail.com', '555-0105', '2026-03-16 16:45:00'),
(6, 'Fernando', 'Silva', 'fer.s@mail.com', '555-0106', '2026-02-28 08:00:00'),
(7, 'Gabriela', 'Cruz', 'gabi.c@mail.com', '555-0107', '2026-03-05 13:10:00'),
(8, 'Hugo', 'Diaz', 'hugo.d@mail.com', NULL, '2026-03-12 17:55:00'),
(9, 'Isabel', 'Ortiz', 'isa.o@mail.com', '555-0109', '2026-01-05 12:00:00'),
(10, 'Jorge', 'Rios', 'jorge.r@mail.com', '555-0110', '2026-03-17 09:00:00');

-- CATEGORIES
CREATE TABLE categories (id INT PRIMARY KEY, parent_id INT, name VARCHAR(100), description TEXT);
INSERT INTO categories VALUES
(1, NULL, 'Electronics', 'Gadgets and devices'),
(2, NULL, 'Clothing', 'Apparel and accessories'),
(3, 1, 'Laptops', 'Computers and notebooks'),
(4, 1, 'Smartphones', 'Mobile devices'),
(5, 2, 'Menswear', 'Men clothing'),
(6, 5, 'Shirts', 'T-shirts and button-downs'),
(7, 5, 'Pants', 'Jeans and trousers'),
(8, 2, 'Womenswear', 'Women clothing'),
(9, 8, 'Dresses', 'Casual and formal dresses'),
(10, NULL, 'Home & Garden', 'Furniture and tools');

-- PRODUCTS
CREATE TABLE products (id INT PRIMARY KEY, category_id INT, name VARCHAR(255), price DECIMAL(10, 2), stock_quantity INT, sku VARCHAR(50) UNIQUE, is_active BOOLEAN);
INSERT INTO products VALUES
(1, 3, 'Pro Laptop 15', 1299.99, 15, 'PROD-123', TRUE),
(2, 4, 'Smartphone X', 799.50, 0, 'SKU-002', TRUE),
(3, 6, 'Cotton T-Shirt', 19.99, 100, 'SKU-003', TRUE),
(4, 7, 'Denim Jeans', 49.99, 50, 'SKU-004', TRUE),
(5, 9, 'Summer Dress', 39.99, 0, 'SKU-005', FALSE),
(6, 3, 'Basic Laptop 13', 599.00, 30, 'SKU-006', TRUE),
(7, 10, 'Garden Shovel', 25.00, 20, 'SKU-007', TRUE),
(8, 10, 'Table Lamp', 35.50, 10, 'SKU-008', FALSE),
(9, 4, 'Budget Phone', 299.00, 0, 'SKU-009', TRUE),
(10, 6, 'Polo Shirt', 29.99, 45, 'SKU-010', TRUE);

-- ORDERS
CREATE TABLE orders (id INT PRIMARY KEY, user_id INT, order_date TIMESTAMP, status VARCHAR(50), total_amount DECIMAL(10, 2));
INSERT INTO orders VALUES
(1, 1, '2024-05-10 10:00:00', 'cancelled', 1299.99),
(2, 1, '2026-03-17 08:30:00', 'pending', 49.99),
(3, 2, '2026-03-16 14:00:00', 'completed', 799.50),
(4, 3, '2024-11-20 09:15:00', 'cancelled', 39.99),
(5, 4, '2026-03-17 11:20:00', 'completed', 599.00),
(6, 5, '2026-02-28 16:45:00', 'completed', 19.99),
(7, 6, '2026-03-15 08:00:00', 'pending', 25.00),
(8, 7, '2026-03-17 13:10:00', 'pending', 35.50),
(9, 8, '2024-01-12 17:55:00', 'completed', 299.00),
(10, 9, '2026-03-17 12:00:00', 'pending', 29.99);

-- ORDER_ITEMS
CREATE TABLE order_items (id INT PRIMARY KEY, order_id INT, product_id INT, quantity INT, unit_price DECIMAL(10, 2));
INSERT INTO order_items VALUES
(1, 1, 1, 1, 1299.99),
(2, 2, 4, 1, 49.99),
(3, 3, 2, 1, 799.50),
(4, 4, 5, 1, 39.99),
(5, 5, 6, 1, 599.00),
(6, 6, 3, 1, 19.99),
(7, 7, 7, 1, 25.00),
(8, 8, 8, 1, 35.50),
(9, 9, 9, 1, 299.00),
(10, 10, 10, 1, 29.99);

-- REVIEWS
CREATE TABLE reviews (id INT PRIMARY KEY, product_id INT, user_id INT, rating INT, comment TEXT, created_at TIMESTAMP);
INSERT INTO reviews VALUES
(1, 1, 2, 5, 'Amazing laptop!', '2026-03-15 10:00:00'),
(2, 2, 3, 1, 'Battery drains too fast.', '2026-02-20 14:30:00'),
(3, 3, 4, 4, 'Good quality cotton.', '2026-03-10 09:15:00'),
(4, 1, 5, 5, 'Best purchase ever.', '2026-03-16 11:20:00'),
(5, 5, 6, 2, 'Size runs small.', '2026-01-20 16:45:00'),
(6, 6, 7, 4, 'Great value for money.', '2026-02-28 08:00:00'),
(7, 7, 8, 5, 'Sturdy and reliable.', '2026-03-05 13:10:00'),
(8, 8, 9, 1, 'Arrived broken.', '2026-03-12 17:55:00'),
(9, 9, 10, 3, 'Its okay for the price.', '2026-01-05 12:00:00'),
(10, 10, 1, 5, 'Fits perfectly.', '2026-03-17 09:00:00');

-- SHIPPING_DETAILS
CREATE TABLE shipping_details (id INT PRIMARY KEY, order_id INT, address_line1 VARCHAR(255), city VARCHAR(100), state VARCHAR(100), postal_code VARCHAR(20), tracking_number VARCHAR(100), carrier VARCHAR(50));
INSERT INTO shipping_details VALUES
(1, 1, '123 Main St', 'Bogota', 'Cundinamarca', '110111', 'TRK-001', 'FedEx'),
(2, 2, '456 Elm St', 'Medellin', 'Antioquia', '050001', 'TRK-002', 'UPS'),
(3, 3, '789 Oak Ave', 'Cali', 'Valle', '760001', 'TRK-003', 'FedEx'),
(4, 4, '321 Pine Rd', 'Cartagena', 'Bolivar', '130001', 'TRK-004', 'DHL'),
(5, 5, '654 Cedar Ln', 'Barranquilla', 'Atlantico', '080001', 'TRK-005', 'FedEx'),
(6, 6, '987 Birch Blvd', 'Bucaramanga', 'Santander', '680001', 'TRK-006', 'UPS'),
(7, 7, '147 Walnut St', 'Pereira', 'Risaralda', '660001', 'TRK-007', 'FedEx'),
(8, 8, '258 Cherry Ct', 'Manizales', 'Caldas', '170001', 'TRK-008', 'DHL'),
(9, 9, '369 Spruce Way', 'Santa Marta', 'Magdalena', '470001', 'TRK-009', 'FedEx'),
(10, 10, '741 Ash Dr', 'Cucuta', 'Norte de Santander', '540001', 'TRK-010', 'UPS');

-- PROMOTIONS
CREATE TABLE promotions (id INT PRIMARY KEY, code VARCHAR(50) UNIQUE, discount_percentage DECIMAL(5, 2), start_date DATE, end_date DATE, is_active BOOLEAN);
INSERT INTO promotions VALUES
(1, 'VERANO20', 20.00, '2026-03-01', '2026-03-31', TRUE),
(2, 'WELCOME10', 10.00, '2026-01-01', '2026-12-31', TRUE),
(3, 'FLASH50', 50.00, '2026-03-15', '2026-03-20', TRUE),
(4, 'WINTER30', 30.00, '2025-12-01', '2026-02-28', FALSE),
(5, 'VIP25', 25.00, '2026-01-01', '2026-12-31', TRUE),
(6, 'SPRING15', 15.00, '2026-03-20', '2026-06-20', FALSE),
(7, 'BF2025', 40.00, '2025-11-25', '2025-11-30', FALSE),
(8, 'CYBER2025', 45.00, '2025-12-01', '2025-12-02', FALSE),
(9, 'FREESHIP', 100.00, '2026-03-01', '2026-03-31', TRUE),
(10, 'HALLOWEEN', 15.00, '2025-10-25', '2025-10-31', FALSE);
"""

# Run the setup SQL
con.execute(setup_sql)
print("Database successfully created and populated with 10 rows per table in DuckDB!")

# Query test
display(con.execute("SELECT * FROM users LIMIT 5;").df())

Database successfully created and populated with 10 rows per table in DuckDB!


,id,first_name,last_name,email,phone,created_at
0,1,Ana,Gomez,cliente@example.com,555-0101,2026-02-15 10:00:00
1,2,Carlos,Ruiz,carlos.r@mail.com,None,2026-03-01 14:30:00
2,3,Beatriz,Perez,bea.p@mail.com,555-0103,2026-03-10 09:15:00
3,4,David,Lopes,david.l@mail.com,None,2026-01-20 11:20:00
4,5,Elena,Torres,elena.t@mail.com,555-0105,2026-03-16 16:45:00


In [ ]:
import os
import getpass
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

# 1. API Key
api_key = getpass.getpass("Introduce tu OpenAI API Key: ")
os.environ["OPENAI_API_KEY"] = api_key

# 2. RAG Esquemas de tablas
table_schemas_docs = [
    Document(
        page_content="Tabla: users\nDescripción: Almacena información de cuentas de clientes, incluyendo detalles de contacto y fecha de registro.\nEsquema: id INT PRIMARY KEY, first_name VARCHAR(50), last_name VARCHAR(50), email VARCHAR(100) UNIQUE, phone VARCHAR(20), created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP",
        metadata={"table_name": "users"}
    ),
    Document(
        page_content="Tabla: categories\nDescripción: Categorías de productos para organizar el catálogo jerárquicamente.\nEsquema: id INT PRIMARY KEY, parent_id INT, name VARCHAR(100), description TEXT",
        metadata={"table_name": "categories"}
    ),
    Document(
        page_content="Tabla: products\nDescripción: Detalles sobre artículos a la venta, precios, vinculación de categorías e inventario.\nEsquema: id INT PRIMARY KEY, category_id INT, name VARCHAR(255), price DECIMAL(10, 2), stock_quantity INT, sku VARCHAR(50) UNIQUE, is_active BOOLEAN",
        metadata={"table_name": "products"}
    ),
    Document(
        page_content="Tabla: orders\nDescripción: Datos transaccionales de alto nivel para compras de clientes, estado del pedido y costo total.\nEsquema: id INT PRIMARY KEY, user_id INT, order_date TIMESTAMP, status VARCHAR(50), total_amount DECIMAL(10, 2)",
        metadata={"table_name": "orders"}
    ),
    Document(
        page_content="Tabla: order_items\nDescripción: Tabla de unión que detalla productos específicos comprados en un pedido, cantidades y precios cerrados.\nEsquema: id INT PRIMARY KEY, order_id INT, product_id INT, quantity INT, unit_price DECIMAL(10, 2)",
        metadata={"table_name": "order_items"}
    ),
    Document(
        page_content="Tabla: reviews\nDescripción: Calificaciones de clientes y comentarios escritos para productos específicos.\nEsquema: id INT PRIMARY KEY, product_id INT, user_id INT, rating INT, comment TEXT, created_at TIMESTAMP",
        metadata={"table_name": "reviews"}
    ),
    Document(
        page_content="Tabla: shipping_details\nDescripción: Información logística, números de seguimiento y direcciones de entrega vinculadas a un pedido.\nEsquema: id INT PRIMARY KEY, order_id INT, address_line1 VARCHAR(255), city VARCHAR(100), state VARCHAR(100), postal_code VARCHAR(20), tracking_number VARCHAR(100), carrier VARCHAR(50)",
        metadata={"table_name": "shipping_details"}
    ),
    Document(
        page_content="Tabla: promotions\nDescripción: Códigos de descuento, porcentajes de oferta y rangos de fechas válidos para promociones de la tienda.\nEsquema: id INT PRIMARY KEY, code VARCHAR(50) UNIQUE, discount_percentage DECIMAL(5, 2), start_date DATE, end_date DATE, is_active BOOLEAN",
        metadata={"table_name": "promotions"}
    )
]

# Inicializar Base de Datos Vectorial
vector_db = FAISS.from_documents(table_schemas_docs, OpenAIEmbeddings())
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

# Diccionario de ejemplos de consultas SQL
query_examples = {
    "users": [
        {"description": "Buscar un usuario por correo electrónico.", "sql": "SELECT * FROM users WHERE email = 'cliente@example.com';"},
        {"description": "Contar nuevos usuarios en el último mes.", "sql": "SELECT COUNT(*) FROM users WHERE created_at >= CURRENT_DATE - INTERVAL '1 month';"},
        {"description": "Buscar usuarios sin número de teléfono.", "sql": "SELECT id, first_name, last_name FROM users WHERE phone IS NULL;"}
    ],
    "categories": [
        {"description": "Obtener todas las categorías principales (sin padre).", "sql": "SELECT name FROM categories WHERE parent_id IS NULL;"},
        {"description": "Listar todas las subcategorías para el ID de categoría 5.", "sql": "SELECT name, description FROM categories WHERE parent_id = 5;"}
    ],
    "products": [
        {"description": "Listar productos inactivos o sin stock.", "sql": "SELECT name, sku FROM products WHERE stock_quantity = 0 OR is_active = FALSE;"},
        {"description": "Encontrar el producto más caro en una categoría.", "sql": "SELECT name, price FROM products WHERE category_id = 3 ORDER BY price DESC LIMIT 1;"},
        {"description": "Calcular el valor total del inventario para un producto.", "sql": "SELECT name, (price * stock_quantity) AS total_value FROM products WHERE sku = 'PROD-123';"}
    ],
    "orders": [
        {"description": "Obtener pedidos pendientes para un usuario específico.", "sql": "SELECT id, order_date, total_amount FROM orders WHERE user_id = 123 AND status = 'pending';"},
        {"description": "Calcular los ingresos totales de hoy.", "sql": "SELECT SUM(total_amount) AS revenue FROM orders WHERE DATE(order_date) = CURRENT_DATE AND status != 'cancelled';"},
        {"description": "Contar pedidos cancelados en 2024.", "sql": "SELECT COUNT(*) FROM orders WHERE status = 'cancelled' AND EXTRACT(YEAR FROM order_date) = 2024;"}
    ],
    "order_items": [
        {"description": "Ver productos y cantidades para un pedido específico.", "sql": "SELECT product_id, quantity, unit_price FROM order_items WHERE order_id = 456;"},
        {"description": "Calcular la cantidad total vendida de un producto específico.", "sql": "SELECT SUM(quantity) AS total_sold FROM order_items WHERE product_id = 789;"},
        {"description": "Identificar qué pedido compró la mayor cantidad de un solo artículo.", "sql": "SELECT order_id, quantity FROM order_items ORDER BY quantity DESC LIMIT 1;"}
    ],
    "reviews": [
        {"description": "Obtener todas las reseñas de 5 estrellas para un producto.", "sql": "SELECT comment, user_id FROM reviews WHERE product_id = 999 AND rating = 5;"},
        {"description": "Calcular la calificación promedio de un producto.", "sql": "SELECT AVG(rating) AS average_rating FROM reviews WHERE product_id = 111;"},
        {"description": "Buscar usuarios que dejaron reseñas de 1 estrella el mes pasado.", "sql": "SELECT user_id, comment FROM reviews WHERE rating = 1 AND created_at >= CURRENT_DATE - INTERVAL '1 month';"}
    ],
    "shipping_details": [
        {"description": "Buscar dirección de envío y seguimiento para un pedido.", "sql": "SELECT address_line1, city, tracking_number, carrier FROM shipping_details WHERE order_id = 456;"},
        {"description": "Contar paquetes enviados a través de FedEx.", "sql": "SELECT COUNT(*) FROM shipping_details WHERE carrier = 'FedEx';"}
    ],
    "promotions": [
        {"description": "Verificar si un código de descuento está activo y es válido hoy.", "sql": "SELECT discount_percentage FROM promotions WHERE code = 'VERANO20' AND is_active = TRUE AND CURRENT_DATE BETWEEN start_date AND end_date;"},
        {"description": "Listar promociones que vencen en los próximos 7 días.", "sql": "SELECT code, end_date FROM promotions WHERE end_date BETWEEN CURRENT_DATE AND CURRENT_DATE + INTERVAL '7 days';"}
    ]
}

# 3. Tools
@tool
def search_schema(query: str) -> str:
    """Busca en la base de datos vectorial los esquemas de tablas SQL relevantes basados en la pregunta del usuario."""
    docs = retriever.invoke(query)
    return "\n\n".join([f"Tabla Encontrada: {d.metadata['table_name']}\n{d.page_content}" for d in docs])

@tool
def get_sql_examples(table_name: str) -> str:
    """Obtiene ejemplos de consultas SQL para una tabla específica. Pasa siempre el nombre exacto de la tabla (ej. 'users', 'orders')."""
    examples = query_examples.get(table_name.lower())
    if not examples:
        return f"No se encontraron ejemplos para la tabla '{table_name}'."

    formatted_examples = [f"- {ex['description']}\n  {ex['sql']}" for ex in examples]
    return "\n".join(formatted_examples)

@tool
def execute_sql(query: str) -> str:
    """
    Ejecuta una consulta SQL generada en la base de datos DuckDB.
    Pasa ÚNICAMENTE la cadena de texto de la consulta SQL pura.
    Devuelve los resultados de la consulta en formato de tabla Markdown, o un mensaje de error si la consulta falla.
    """
    try:
        # Ejecuta la consulta y la convierte en un DataFrame de pandas
        result_df = con.execute(query).df()

        if result_df.empty:
            return "La consulta se ejecutó correctamente, pero no devolvió resultados (0 filas)."

        # Convierte el DataFrame a una tabla Markdown para que el agente la lea fácilmente
        return result_df.to_markdown(index=False)
    except Exception as e:
        return f"Error de ejecución SQL: {e}\nPor favor, revisa tu sintaxis SQL y vuelve a intentarlo."

# 4. Agente
system_prompt = (
    "Eres un Asistente experto en SQL que interactúa con una base de datos de comercio electrónico. "
    "Sigue estos pasos estrictamente:\n"
    "1. Usa 'search_schema' para encontrar los esquemas de tabla correctos.\n"
    "2. Usa 'get_sql_examples' con los nombres de las tablas para buscar consultas de referencia.\n"
    "3. Escribe la consulta SQL exacta que resuelve la solicitud.\n"
    "4. IMPORTANTE: Pasa la consulta SQL a la herramienta 'execute_sql' para ejecutarla en la base de datos.\n"
    "5. Basado en los resultados devueltos por 'execute_sql', responde a la pregunta original del usuario de forma natural, mostrando los datos obtenidos.\n"
    "Nota: Si el usuario pide varias cosas (ej. ingresos y productos sin stock), puedes usar la herramienta 'execute_sql' varias veces con diferentes consultas."
)


model = ChatOpenAI(model="gpt-4.1-nano", temperature=0.1)
tools = [search_schema, get_sql_examples, execute_sql]
memory = MemorySaver()

app = create_react_agent(
    model=model,
    tools=tools,
    prompt=system_prompt,
    checkpointer=memory
)

Introduce tu OpenAI API Key: ··········


/tmp/ipykernel_10419/2497268533.py:148: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  app = create_react_agent(


In [ ]:
config = {"configurable": {"thread_id": "estudiante_1"}}
user_input = "lista todos los productos que actualmente tienen 0 existencias en el inventario"

print(f"--- Procesando consulta para el thread_id: {config['configurable']['thread_id']} ---\n")

for event in app.stream({"messages": [("user", user_input)]}, config, stream_mode="values"):
    if "messages" in event:
        event["messages"][-1].pretty_print()

--- Procesando consulta para el thread_id: estudiante_1 ---

================================ Human Message =================================

lista todos los productos que actualmente tienen 0 existencias en el inventario
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_IHsa43IseVXalBXJSsVCVImy)
 Call ID: call_IHsa43IseVXalBXJSsVCVImy
  Args:
    query: SELECT name, sku FROM products WHERE stock_quantity = 0;
================================= Tool Message =================================
Name: execute_sql

| name         | sku     |
|:-------------|:--------|
| Smartphone X | SKU-002 |
| Summer Dress | SKU-005 |
| Budget Phone | SKU-009 |
================================== Ai Message ==================================

Los productos que actualmente tienen 0 existencias en el inventario son Smartphone X, Summer Dress y Budget Phone.
